# Prédire le temps de réaction à partir de l'EEG

Projet IODAA, EEG Foundation Challenge (challenge 1). Données : HBN-EEG, releases mini R1 à R5 (100 sujets), tâche Contrast Change Detection.

## 1. Le problème

Les données viennent du **Healthy Brain Network (HBN-EEG)** : des enfants et des adolescents (5 à 21 ans) enregistrés en EEG pendant plusieurs tâches. Le signal a **129 canaux**, il est filtré puis ramené à **100 Hz**, et stocké en fichiers `.bdf` (un dossier `sub-XXXX/eeg/` par sujet).

J'utilise les **releases mini R1 à R5** : 20 sujets par release, soit 100 sujets.

**La tâche : Contrast Change Detection.** Le sujet regarde deux disques rayés qui clignotent. Après un délai aléatoire, le contraste de l'un monte et celui de l'autre baisse. Le sujet appuie à gauche ou à droite pour désigner le disque dont le contraste domine, puis reçoit un feedback (smiley).

**Ce qu'on prédit.** Le **temps de réaction** en secondes, c'est-à-dire le délai entre le changement de contraste et l'appui. L'entrée est une fenêtre EEG de 2 s, soit un tableau `(129 canaux, 200 points)`. La métrique est la RMSE.

**La difficulté.** Le modèle doit marcher sur des sujets qu'il n'a jamais vus. Les sujets sont très différents entre eux et l'EEG est très bruité. La validation croisée regroupe donc les essais par sujet (`GroupKFold`).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, mne
import matplotlib.pyplot as plt

from scipy.signal import welch
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

mne.set_log_level("WARNING")

## 2. Les données

On commence par lister tous les fichiers EEG des 5 releases : un enregistrement par sujet et par tâche (et par run pour certaines tâches).

In [ ]:
inv = pd.DataFrame({"path": sorted(Path("data").glob("R*_mini_L100_bdf/sub-*/eeg/*_eeg.bdf"))})
inv["release"] = inv.path.map(lambda p: p.parts[1])
inv["subject"] = inv.path.map(lambda p: p.name.split("_")[0])
inv["task"] = inv.path.map(lambda p: p.name.split("task-")[1].split("_")[0])

print(inv.groupby(["release", "task"]).size().unstack())     # enregistrements par release et par tâche
ccd = inv[inv.task == "contrastChangeDetection"]
print(ccd.groupby("subject").size().value_counts())          # nombre de runs CCD par sujet

La tâche étudiée est `contrastChangeDetection` (CCD). Chaque enregistrement a un fichier `_events.tsv` avec les événements datés, un `_channels.tsv` et un `.json` de métadonnées.

Chaque release a un fichier `participants.tsv` : une ligne par sujet. Je les assemble en un seul tableau.

In [ ]:
L = []
for release in sorted(Path("data").glob("R*_mini_L100_bdf")):
    p = pd.read_csv(release / "participants.tsv", sep="\t")
    p["release"] = release.name
    L.append(p)
participants = pd.concat(L, ignore_index=True)

print(participants.shape)                                                  # (100, 25)
print(participants.groupby("release")["age"].agg(["count", "mean", "min", "max"]))
print(participants["sex"].value_counts())

Le tableau `participants` a 100 lignes : 20 sujets par release.

- `participant_id` : la clé pour relier un sujet à son dossier `sub-...`.
- `sex`, `age`, `ehq_total` (latéralité).
- `p_factor`, `attention`, `internalizing`, `externalizing` : scores de psychopathologie (cibles de l'autre volet du challenge).
- Une colonne par tâche, qui dit si l'enregistrement est disponible.

L'âge va de 5,4 à 18,9 ans, avec une moyenne d'environ 10 ans dans chaque release. L'âge et le sexe ne font pas partie des entrées du challenge officiel : je m'en sers pour comprendre les données et comme référence.

## 3. Calcul du temps de réaction

Je pars d'un seul fichier : le run 1 du premier sujet de R5. Chaque ligne de `_events.tsv` est un événement, avec sa date en secondes (`onset`). Un essai se déroule ainsi : le contraste change (`left_target` ou `right_target`), puis le sujet appuie (`left_buttonPress` ou `right_buttonPress`).

**Le temps de réaction** est la durée entre le changement de contraste et l'appui.

**Le calcul.**
1. Je garde seulement les lignes `target` et `buttonPress` (46 lignes sur 75).
2. Je calcule l'écart de temps avec la ligne précédente (`diff()` sur `onset`).
3. Je garde les lignes d'appui : sur ces lignes, l'écart est le délai avec le `target` juste avant, donc le temps de réaction.
4. J'enlève les appuis dont le feedback est `non_target` (1 cas), qui ne sont pas des réponses valides.

**Vérification.** Chaque appui retenu est précédé d'un `target`. Le premier temps vaut 2,110 s, ce qui correspond à mon calcul à la main (112,172 − 110,062). On obtient 21 temps de réaction, entre 1,22 s et 2,36 s.

In [ ]:
chemin_ev = "data/R5_mini_L100_bdf/sub-NDARAH793FBF/eeg/sub-NDARAH793FBF_task-contrastChangeDetection_run-1_events.tsv"
ev = pd.read_csv(chemin_ev, sep="\t")
print(ev.head(30))

masque = ev["value"].str.endswith("target") | ev["value"].str.endswith("buttonPress")
print("lignes gardées :", masque.sum())
sub = ev.loc[masque].copy()
sub["delta"] = sub["onset"].diff()

masque_1 = sub["value"].str.endswith("buttonPress") & (sub["feedback"] != "non_target")
print(sub["value"].shift(1)[masque_1].value_counts())    # chaque appui est précédé d'un target
print(sub.loc[masque_1, "delta"].head(30))

La fonction `f_temps_reaction(chemin)` refait ce calcul pour n'importe quel fichier d'événements. Elle renvoie une ligne par essai valide :
- `rt` : le temps de réaction ;
- `onset_stim` : la date du stimulus (elle servira à découper le signal) ;
- `run` et `subject`, lus dans le nom du fichier.

In [ ]:
def f_temps_reaction(chemin):
    ev = pd.read_csv(chemin, sep="\t")
    morceaux = Path(chemin).name.split("_")
    masque = ev["value"].str.endswith("target") | ev["value"].str.endswith("buttonPress")
    sub = ev.loc[masque].copy()
    sub["subject"] = morceaux[0]
    sub["run"] = morceaux[2]
    sub["rt"] = sub["onset"].diff()
    sub["onset_stim"] = sub["onset"].shift(1)
    masque_1 = sub["value"].str.endswith("buttonPress") & (sub["feedback"] != "non_target")
    return sub.loc[masque_1, ["rt", "onset_stim", "run", "subject"]]

res = f_temps_reaction(chemin_ev)
print(res.head(), len(res))      # 21 essais

On applique la fonction à tous les fichiers d'événements CCD des 5 releases, et on ajoute le nom de la release.

In [ ]:
L = []
for release in sorted(Path("data").glob("R*_mini_L100_bdf")):
    for chemin in sorted(release.glob("sub-*/eeg/*contrastChangeDetection*_events.tsv")):
        d = f_temps_reaction(chemin)
        d["release"] = release.name
        L.append(d)

rt_all = pd.concat(L, ignore_index=True)
print(rt_all.shape)                                  # (6813, 5)
print(rt_all["subject"].nunique(), "sujets")

On obtient 6 813 essais pour 100 sujets. Avant de s'en servir, on regarde s'il y a des valeurs bizarres.

In [ ]:
print(rt_all.groupby("subject").size().sort_values().head())    # sujets avec le moins d'essais
print(rt_all["rt"].describe())
plt.hist(rt_all["rt"], bins=30)
plt.xlabel("temps de réaction (s)"); plt.show()

**Les essais très rapides.** Quelques temps de réaction sont beaucoup trop courts pour être une vraie réaction (de l'ordre de 50 ms). En regardant les événements d'un de ces essais, il n'y a pas d'erreur dans les données : c'est probablement un appui anticipé.

In [ ]:
print(rt_all[rt_all["rt"] < 0.5].head())

chemin_sub = "data/R5_mini_L100_bdf/sub-NDARBE091BGD/eeg/sub-NDARBE091BGD_task-contrastChangeDetection_run-1_events.tsv"
print(pd.read_csv(chemin_sub, sep="\t").head(50))

for seuil in [0.2, 0.3, 0.5, 0.7]:
    print(seuil, (rt_all["rt"] < seuil).sum())

**Le seuil.** D'après l'histogramme, j'écarte les temps de réaction inférieurs à 0,5 s. Cela retire 137 essais (2 %) et laisse 6 676 essais, dans la variable `rt_all_s`.

In [ ]:
rt_all_s = rt_all[rt_all["rt"] > 0.5]
print(rt_all_s.shape)                                # (6676, 5)
print(rt_all_s.groupby("release")["subject"].nunique())
rt_all_s.describe()

## 4. Découpage des fenêtres EEG

Je teste d'abord sur le run 1 du premier sujet de R5. Le fichier `.bdf` se lit avec `mne.io.read_raw_bdf` et `get_data()` donne un tableau `(129, nombre de points)`.

Une fenêtre commence **0,5 s après le stimulus** et dure 200 points, soit 2 s à 100 Hz. L'indice de départ est `int((onset_stim + 0,5) × fréquence d'échantillonnage)`.

In [ ]:
raw = mne.io.read_raw_bdf("data/R5_mini_L100_bdf/sub-NDARAH793FBF/eeg/sub-NDARAH793FBF_task-contrastChangeDetection_run-1_eeg.bdf", preload=True)
sfreq = raw.info["sfreq"]
tab = raw.get_data()
print(sfreq, tab.shape)                              # 100.0, (129, nombre de points)

masque = (rt_all_s["subject"] == "sub-NDARAH793FBF") & (rt_all_s["run"] == "run-1")
ligne = rt_all_s[masque].iloc[0]
debut = int((ligne["onset_stim"] + 0.5) * sfreq)
print(tab[:, debut:debut + 200].shape)               # (129, 200)

La fonction `f_fenetre(chemin, onset_stim)` fait ce découpage pour tout un enregistrement.

- **Entrée** : le chemin du fichier `.bdf` et la liste des dates de stimulus de ce run.
- **Chargement** : le fichier est lu une seule fois, pas une fois par essai.
- **Sortie** : un tableau `(n_essais, 129, 200)`, en `float32` pour économiser la mémoire.

Vérification sur le run 1 du premier sujet : 21 essais, donc `(21, 129, 200)`.

In [ ]:
def f_fenetre(chemin, onset_stim):
    raw = mne.io.read_raw_bdf(chemin, preload=True)
    tab = raw.get_data()
    sfreq = raw.info["sfreq"]
    L = []
    for k in onset_stim:
        debut = int((k + 0.5) * sfreq)
        fin = debut + 200
        L.append(tab[:, debut:fin])
    return np.stack(L).astype(np.float32)

subject, run = "sub-NDARAH793FBF", "run-1"
masque = (rt_all_s["subject"] == subject) & (rt_all_s["run"] == run)
chemin = f"data/R5_mini_L100_bdf/{subject}/eeg/{subject}_task-contrastChangeDetection_{run}_eeg.bdf"
print(f_fenetre(chemin, rt_all_s.loc[masque, "onset_stim"]).shape)      # (21, 129, 200)

## 5. Construction du jeu de données

Je rassemble les fenêtres EEG, les temps de réaction, les sujets et les releases de tous les essais valides.

- **Parcours** : `groupby(["release", "subject", "run"])` donne un groupe par enregistrement. Le chemin du fichier dépend de la release. Chaque `.bdf` est chargé une seule fois.
- **Alignement** : les quatre tableaux sont construits dans la même boucle, donc l'essai numéro *k* est le même partout.
- **Résultat** : `X_fen` `(6676, 129, 200)`, `y` `(6676,)`, `groupes` (le sujet de chaque essai) et `releases`.

In [ ]:
L_rt, L_subject, L_release, L_fen = [], [], [], []

for (release, subject, run), groupe in rt_all_s.groupby(["release", "subject", "run"]):
    chemin = f"data/{release}/{subject}/eeg/{subject}_task-contrastChangeDetection_{run}_eeg.bdf"
    L_rt.append(groupe["rt"].to_numpy())
    L_subject.append(groupe["subject"].to_numpy())
    L_release.append(groupe["release"].to_numpy())
    L_fen.append(f_fenetre(chemin, groupe["onset_stim"]))

X_fen = np.concatenate(L_fen)
y = np.concatenate(L_rt)
groupes = np.concatenate(L_subject)
releases = np.concatenate(L_release)
print(X_fen.shape, y.shape, groupes.shape, releases.shape)      # (6676, 129, 200) (6676,) (6676,) (6676,)

**Fenêtres plates.** 63 fenêtres ont un signal constant (écart-type médian des 128 canaux inférieur à 1e-9 V). Elles viennent toutes du même sujet, `sub-NDARCA740UC8`, dont l'enregistrement est vide (écart-type d'environ 5e-18 V sur tous les canaux). Je les retire : il reste **6 613 essais et 99 sujets**.

Je sauvegarde ensuite les tableaux dans `data/` pour ne pas relire les fichiers à chaque session.

In [ ]:
std_fen = np.median(X_fen[:, :128, :].std(axis=2), axis=1)
plat = std_fen < 1e-9
print(plat.sum(), "fenêtres plates")
print(pd.Series(groupes[plat]).value_counts())       # quels sujets sont concernés

X_fen, y, groupes, releases = X_fen[~plat], y[~plat], groupes[~plat], releases[~plat]
print(X_fen.shape, y.shape, groupes.shape, releases.shape)      # (6613, 129, 200) (6613,) (6613,) (6613,)

np.save("data/X_fen_5rel.npy", X_fen)
np.save("data/y_5rel.npy", y)
np.save("data/groupes_5rel.npy", groupes)
np.save("data/releases_5rel.npy", releases)

## 6. Visualisations

### Temps de réaction par sujet

Je trace un boxplot du temps de réaction pour chaque sujet.

- Les sujets ne sont pas au même niveau : les médianes diffèrent beaucoup d'un sujet à l'autre.
- Chaque sujet est aussi très dispersé : les boîtes sont larges.

**Conséquence.** Un modèle pourrait prédire le temps de réaction en reconnaissant le sujet, sans rien apprendre de la réaction elle-même. C'est pourquoi la validation se fait par sujet (`GroupKFold`).

In [ ]:
df = pd.DataFrame({"rt": y, "sujet": groupes})
df.boxplot(column="rt", by="sujet", figsize=(16, 5))
plt.suptitle(""); plt.title("")
plt.xticks([]); plt.xlabel("99 sujets"); plt.ylabel("temps de réaction (s)")
plt.show()

### Qualité des canaux

Pour chaque canal, je calcule l'écart-type du signal dans chaque fenêtre, puis sa moyenne sur toutes les fenêtres, en µV.

- Le canal 128 est plat : c'est le canal de référence, d'après la documentation du challenge.
- Un canal dont l'écart-type est très au-dessus des autres est probablement défectueux ou touché par des artefacts. Je n'ai pas vérifié si ce sont les mêmes canaux chez tous les sujets.

In [ ]:
std_cannaux = X_fen.std(axis=2).mean(axis=0) * 1e6
plt.plot(std_cannaux)
plt.xlabel("canal"); plt.ylabel("écart-type moyen (µV)")
plt.show()

### Temps de réaction et âge

Je calcule le temps de réaction moyen de chaque sujet, que je relie à son âge, puis je trace un nuage de points (une couleur par release).

**Observations**
- Les temps de réaction baissent un peu avec l'âge, mais le nuage est très dispersé : le temps de réaction moyen des sujets va d'environ 1,0 s à 2,0 s, à tous les âges.
- La corrélation est de **−0,31** sur 99 sujets.
- Elle varie beaucoup d'une release à l'autre (20 sujets chacune) : −0,01 pour R1, +0,35 pour R2, −0,57 pour R3, −0,30 pour R4 et −0,85 pour R5.

**Lecture.** Avec 20 sujets, la corrélation est très instable. Mon premier résultat sur R5 seul (−0,82) était en partie dû au hasard des sujets. Il faut des dizaines de sujets pour estimer ce lien.

In [ ]:
rt_sujet = (pd.DataFrame({"subject": groupes, "rt": y, "release": releases})
              .groupby("subject")
              .agg(rt_moyen=("rt", "mean"), n=("rt", "size"), release=("release", "first"))
              .reset_index())

aff = rt_sujet.merge(participants[["participant_id", "age"]],
                     left_on="subject", right_on="participant_id")

print(aff.shape)                                                 # (99, 6)
print(aff[["rt_moyen", "age"]].corr().iloc[0, 1])                # -0.31

plt.figure(figsize=(7, 5))
for rel, g in aff.groupby("release"):
    plt.scatter(g["age"], g["rt_moyen"], label=rel[:2])
plt.xlabel("âge (ans)"); plt.ylabel("RT moyen (s)")
plt.legend(); plt.show()

for rel, g in aff.groupby("release"):
    print(rel, round(g["age"].corr(g["rt_moyen"]), 2))

## 7. Les références à battre

Avant de tester un modèle sur l'EEG, je mesure ce que donnent deux modèles très simples. Tous les scores de ce notebook utilisent le même `GroupKFold` à 5 plis, avec le sujet comme groupe : un sujet n'est jamais à la fois dans l'apprentissage et dans le test. La métrique est la RMSE en secondes.

- **Moyenne** : le modèle prédit toujours la moyenne des temps de réaction d'apprentissage (`DummyRegressor`).
- **Âge seul** : le modèle prédit le temps de réaction à partir de l'âge du sujet (`LinearRegression`).

| Modèle | RMSE moyenne | RMSE par pli |
|---|---|---|
| Moyenne | 0,3431 | 0,331 ; 0,348 ; 0,357 ; 0,345 ; 0,334 |
| Âge seul | 0,3355 | 0,330 ; 0,331 ; 0,347 ; 0,349 ; 0,321 |

**Lecture.** L'âge apporte un petit gain, environ 2 %. Ce sont les deux chiffres à battre. L'écart entre les plis (de 0,32 à 0,36) est plus grand que la plupart des différences entre modèles : une différence de 0,001 est du bruit.

In [ ]:
age_par_sujet = participants.set_index("participant_id")["age"]
age_fen = pd.Series(groupes).map(age_par_sujet).to_numpy().reshape(-1, 1)
print(np.isnan(age_fen).sum())                      # attendu : 0

gkf = GroupKFold(n_splits=5)

for nom, mod in [("moyenne", DummyRegressor()), ("âge seul", LinearRegression())]:
    s = -cross_val_score(mod, age_fen, y, groups=groupes, cv=gkf,
                         scoring="neg_root_mean_squared_error")
    print(nom, s.mean().round(4), s.round(3))

## 8. Première famille de features : puissance par bande

Chaque fenêtre `(129 canaux, 200 points)` est résumée par sa puissance dans 5 bandes de fréquence, calculée par canal avec `scipy.signal.welch` (`fs=100`, `nperseg=100`, donc une résolution de 1 Hz).

| Bande | Fréquences |
|---|---|
| delta | 1 à 4 Hz |
| theta | 4 à 8 Hz |
| alpha | 8 à 13 Hz |
| beta | 13 à 30 Hz |
| gamma | 30 à 50 Hz |

Pour chaque bande, je moyenne la puissance sur ses fréquences. J'obtiens `(129, 5)` par fenêtre, aplati en un vecteur de 645 variables. Pour tous les essais, le tableau `puissance` fait `(6613, 645)`.

In [ ]:
def f_puissance(fenetre):
    f, psd = welch(fenetre, fs=100, nperseg=100, axis=-1)
    bandes = {"delta": (f >= 1) & (f < 4), "theta": (f >= 4) & (f < 8), "alpha": (f >= 8) & (f < 13),
              "beta": (f >= 13) & (f < 30), "gamma": (f >= 30) & (f <= 50)}
    puissance = [psd[:, m].mean(axis=1) for m in bandes.values()]
    return np.stack(puissance, axis=1).ravel()

print(f_puissance(X_fen[0]).shape)                   # (645,)
puissance = np.stack([f_puissance(f) for f in X_fen])
print(puissance.shape)                               # (6613, 645)

### Nettoyage des puissances

**Les zéros.** Certains canaux sont parfois plats (signal constant), ce qui donne une puissance exactement nulle. Je regarde où ils sont, par canal.

In [ ]:
print((puissance == 0).sum(), "zéros exacts")
nb_zeros = (puissance == 0).sum(axis=0).reshape(129, 5)[:, 0]      # par canal
print({canal: int(n) for canal, n in enumerate(nb_zeros) if n > 0})

Le canal 128 (la référence) est plat sur toutes les fenêtres : je l'enlève, ce qui laisse **640 variables**. Les autres zéros sont des canaux déconnectés sur quelques fenêtres : je les remplace par des **NaN**.

**Le `log10`.** Les puissances s'étalent sur beaucoup d'ordres de grandeur. Je passe au `log10` pour resserrer l'échelle.

**Les valeurs quasi nulles.** Après le `log10`, 35 626 valeurs sont inférieures à −20, alors qu'une puissance d'EEG normale donne autour de −12. Ce sont des canaux presque plats, mais pas exactement nuls, donc le filtre sur les zéros ne les attrape pas. Elles faussent le `StandardScaler` et Ridge. Je les remplace aussi par des NaN.

**Résultat.** 46 809 NaN sur 4,2 millions de valeurs (environ 1 %). Les valeurs restantes vont de −20 à −2, avec une médiane à −11,7. Les NaN sont remplacés plus tard par la moyenne de la colonne (`SimpleImputer`) dans la `Pipeline`.

In [ ]:
puissance_640 = puissance[:, :640].copy()            # on enlève le canal 128 (référence)
puissance_640[puissance_640 == 0] = np.nan
puissance_log = np.log10(puissance_640)
print(np.isnan(puissance_log).sum())                 # zéros devenus NaN : 11183
print((puissance_log < -20).sum())                   # valeurs quasi nulles : 35626

puissance_log[puissance_log < -20] = np.nan
print(np.isnan(puissance_log).sum())                 # NaN au total : 46809
print(np.nanmin(puissance_log), np.nanmedian(puissance_log), np.nanmax(puissance_log))

## 9. Modèles sur la puissance par bande

Le modèle de base est une `Pipeline` : imputation des NaN par la moyenne, `StandardScaler`, puis Ridge. Pour ne pas répéter le code, je définis deux petites fonctions :
- `ridge_pipe(alpha)` crée la `Pipeline` ;
- `balayage_alpha(X, cible, alphas)` teste plusieurs `alpha` avec le `GroupKFold` et renvoie un tableau (RMSE moyen, pire pli, meilleur pli).

In [ ]:
def ridge_pipe(alpha):
    return Pipeline([("imputer", SimpleImputer(strategy="mean")),
                     ("scaler", StandardScaler()),
                     ("reg", Ridge(alpha=alpha))])

def balayage_alpha(X, cible, alphas):
    lignes = []
    for alpha in alphas:
        s = -cross_val_score(ridge_pipe(alpha), X, cible, groups=groupes, cv=gkf,
                             scoring="neg_root_mean_squared_error")
        lignes.append({"alpha": alpha, "rmse_moyen": s.mean(),
                       "rmse_pire": s.max(), "rmse_meilleur": s.min()})
    return pd.DataFrame(lignes).round(4)

balayage_alpha(puissance_log, y, [1e2, 1e3, 1e4, 1e5, 1e6])

### Ridge

| alpha | RMSE moyenne | pire pli | meilleur pli |
|---|---|---|---|
| 100 | 0,3755 | 0,3952 | 0,3564 |
| 1 000 | 0,3550 | 0,3742 | 0,3416 |
| 10 000 | 0,3444 | 0,3652 | 0,3309 |
| 100 000 | 0,3422 | 0,3621 | 0,3282 |
| 1 000 000 | 0,3420 | 0,3595 | 0,3293 |

Le score s'améliore quand `alpha` augmente et s'arrête au niveau de la moyenne (0,3431). Ridge préfère ignorer les features : elles contiennent très peu de signal linéaire. Il ne bat pas non plus la référence « âge seul » (0,3355).

**Historique.** Sur R5 seul, Ridge donnait des RMSE absurdes (jusqu'à 7 s) à cause de fenêtres et de canaux quasi plats. Le traitement des NaN de la section précédente règle ce problème.

### XGBoost

Pour voir si un modèle non linéaire trouve quelque chose, j'essaie `XGBRegressor`. Il accepte les NaN, donc pas besoin d'imputer.

In [ ]:
mod = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.5, random_state=0)
s = -cross_val_score(mod, puissance_log, y, groups=groupes, cv=gkf,
                     scoring="neg_root_mean_squared_error")
print(s.mean().round(4), s.round(3))                 # 0.3445 [0.331 0.335 0.359 0.35 0.347]

**Résultat.** RMSE moyenne de 0,3445, avec des plis entre 0,331 et 0,359. C'est au niveau de la moyenne (0,3431) : XGBoost ne trouve rien de plus que Ridge.

## 10. Courbe d'apprentissage

Je regarde si le modèle progresse quand on lui donne plus d'essais d'apprentissage, avec `learning_curve` et le même `GroupKFold`. Le modèle est la `Pipeline` avec `alpha = 100 000`. Je trace la RMSE d'apprentissage et de test pour 5 tailles, de 20 % à 100 % des essais disponibles, avec les deux références en pointillé.

Comment la lire :
- si la courbe de test descend vers les références quand on ajoute des essais, plus de données aideraient ;
- si elle reste plate, le blocage vient des features.

In [ ]:
tailles, scores_train, scores_test = learning_curve(
    ridge_pipe(1e5), puissance_log, y, groups=groupes, cv=gkf,
    train_sizes=np.linspace(0.2, 1.0, 5), scoring="neg_root_mean_squared_error")

plt.plot(tailles, (-scores_train).mean(axis=1), label="apprentissage")
plt.plot(tailles, (-scores_test).mean(axis=1), label="test")
plt.axhline(0.3431, linestyle="--", color="gray", label="moyenne")
plt.axhline(0.3355, linestyle=":", color="gray", label="âge seul")
plt.xlabel("nombre d'essais d'apprentissage"); plt.ylabel("RMSE (s)")
plt.legend(); plt.show()

## 11. Deuxième famille de features : le signal dans le temps

La puissance moyenne sur 2 s efface le moment où les choses se passent. Je garde donc l'information temporelle : pour chaque canal, la **moyenne du signal par tranche de 200 ms**. Chaque fenêtre devient `128 canaux × 10 tranches = 1 280` variables (le canal 128 est enlevé).

In [ ]:
X_temps = X_fen[:, :128, :].reshape(-1, 128, 10, 20).mean(axis=3).reshape(-1, 128 * 10)
print(X_temps.shape)                                 # (6613, 1280)

balayage_alpha(X_temps, y, [1e2, 1e4, 1e6, 1e7])

Les résultats de ce balayage sont à lire dans le tableau, à comparer aux références (0,3431 pour la moyenne, 0,3355 pour l'âge).

## 12. Retirer l'effet « sujet »

Le sujet explique environ 20 % de la variance du temps de réaction ; les 80 % restants viennent des variations d'un essai à l'autre chez une même personne. Pour tester s'il reste un signal **à l'intérieur de chaque sujet**, je retire la moyenne du sujet des features et de `y` : le modèle doit alors prédire si un essai est plus rapide ou plus lent que d'habitude pour cette personne.

La référence devient un modèle qui prédit toujours 0 (`DummyRegressor(strategy="constant", constant=0.0)`). Elle vaut **0,3054**.

In [ ]:
def centrer(X_, groupes_):
    X_df = pd.DataFrame(X_)
    return (X_df - X_df.groupby(groupes_).transform("mean")).to_numpy()

y_c = y - pd.Series(y).groupby(groupes).transform("mean").to_numpy()
X_c = centrer(puissance_log, groupes)

s = -cross_val_score(DummyRegressor(strategy="constant", constant=0.0), X_c, y_c,
                     groups=groupes, cv=gkf, scoring="neg_root_mean_squared_error")
print("zéro (baseline)", s.mean().round(4), s.round(3))           # 0.3054

print(balayage_alpha(X_c, y_c, [1e5]))                            # puissance : 0.3055
print(balayage_alpha(centrer(X_temps, groupes), y_c, [1e4, 1e5, 1e6]))   # tranches de temps

| Modèle | Features | RMSE moyenne |
|---|---|---|
| Zéro (référence) | aucune | 0,3054 |
| Ridge (alpha = 1e5) | puissance par bande | 0,3055 |
| Ridge (alpha = 1e4) | tranches de 200 ms | 0,4021 |
| Ridge (alpha = 1e5) | tranches de 200 ms | 0,3282 |
| Ridge (alpha = 1e6) | tranches de 200 ms | 0,3071 |

Aucun modèle ne passe sous la référence. Avec les `alpha` faibles, les tranches de temps font même nettement pire, avec un pli à 0,689 : un sujet aux artefacts de grande amplitude suffit à dérégler le modèle.

## 13. Réseau de neurones sur le signal brut

Je teste aussi **EEGNeX** (braindecode, PyTorch), le réseau du startkit, directement sur les fenêtres `(129, 200)` mises en µV. Ce test est dans un notebook séparé, `eegnex.ipynb`, car il est lent et le kernel avait planté une fois.

Il porte sur un seul pli du `GroupKFold` de R5 (19 sujets, 903 essais d'apprentissage et 240 de validation), avec 10 epochs (AdamW, lr = 1e-3). La RMSE de validation reste entre 0,337 et 0,378, pour une référence de 0,335 (prédire la moyenne du train). Un entraînement de 10 epochs prend environ 21 minutes.

Le réseau ne fait pas mieux que les modèles à features. Ce test n'a qu'un pli, une graine et aucun réglage : il ne prouve pas qu'aucun réseau ne marcherait.

## 14. Bilan

Les tests sont faits sur les releases R1 à R5 : 99 sujets, 6 613 essais. L'évaluation est un `GroupKFold` à 5 plis avec des sujets séparés, et la métrique est la RMSE en secondes.

| Modèle | Features | RMSE moyenne |
|---|---|---|
| Moyenne (référence) | aucune | 0,3431 |
| Régression linéaire | âge seul | 0,3355 |
| Ridge (alpha = 1e6) | puissance par bande | 0,3420 |
| XGBoost | puissance par bande | 0,3445 |

Avec l'effet sujet retiré (référence : prédire 0) :

| Modèle | Features | RMSE moyenne |
|---|---|---|
| Zéro (référence) | aucune | 0,3054 |
| Ridge (alpha = 1e5) | puissance par bande | 0,3055 |
| Ridge (alpha = 1e6) | tranches de 200 ms | 0,3071 |

Sur un seul pli de R5, EEGNeX donne 0,34 à 0,38 contre 0,335 pour la moyenne.

**Conclusion.** Aucun modèle ne bat clairement la moyenne ni l'âge seul. Les features testées (puissance par bande, signal par tranche de 200 ms) ne contiennent pas d'information exploitable sur le temps de réaction, ni entre les sujets, ni à l'intérieur d'un sujet. L'écart entre les plis (de 0,33 à 0,36) est plus grand que les différences entre modèles.

**Ce que j'ai appris.**
- Avec 20 sujets, les résultats sont très instables : la corrélation âge / temps de réaction allait de −0,85 à +0,35 selon la release.
- Les canaux plats et les valeurs quasi nulles peuvent faire exploser un modèle linéaire : il faut les traiter avant le `StandardScaler`.
- Une bonne référence est indispensable : sans la moyenne et l'âge seul, des RMSE de 0,34 auraient pu passer pour un résultat.

**Pistes.** Ajouter d'autres releases (R6 à R11), choisir des canaux (occipitaux, pariétaux) ou des fréquences précises, et utiliser des features centrées sur les événements (type potentiel évoqué).